# Exportable Power BI comparison
Run this notebook from the repository root or its power-bi directory. It builds a native paginated report for My workspace from the same strategy table as the other dashboards. Each page is a data-bound matrix with embedded sample rows; no gateway or remote data connection is required.

In [ ]:
from pathlib import Path
import csv
import xml.etree.ElementTree as ET

here = Path.cwd()
if here.name == "power-bi":
    here = here.parent.parent
data = list(csv.DictReader((here / "results/strategy_cells.csv").open(encoding="utf-8-sig")))
book = ET.parse(here / "dashboards/tableau/gameplay.twb")
names = ["Printed curve", "Deployment routes", "Interaction targets", "Interaction methods", "Recovery routes",
         "Printed card colors", "Commander identity", "Casting-cost emphasis", "Card types", "Creature types",
         "Human-adjusted tribes", "Creature composition", "Mana and access", "Interaction readiness", "Timing and setup"]
assert set(names) == {r["view"] for r in data}
units = {s.get("name")[10:-6]:s.find("layout-options/title/formatted-text").findall("run")[1].text.strip(" |")
         for s in book.findall("worksheets/worksheet") if s.get("name").startswith("Compare | ")}
units.update({"Printed curve":"Nonland library slots by mana value; commanders excluded.",
              "Printed card colors":"Percent of all nonland slots, including the commander; multicolor cards count separately."})
ns = "http://schemas.microsoft.com/sqlserver/reporting/2016/01/reportdefinition"
root = ET.Element("Report", xmlns=ns)
def node(parent, tag, value=None, **attrs):
    e = ET.SubElement(parent, tag, attrs)
    if value is not None:
        e.text = str(value)
    return e
def inch(value):
    return f"{value:.4f}in"
def style(parent, **attrs):
    st = node(parent, "Style")
    for k,v in attrs.items():
        node(st,k,v)
    return st
serial = 0
def box(items, value, font="9pt", ink="#173744", weight="Normal", align="Left", fill=None, pos=None):
    global serial
    serial += 1
    tb = node(items, "Textbox", Name=f"Text{serial}")
    node(tb,"CanGrow","false")
    node(tb,"KeepTogether","true")
    par = node(node(tb,"Paragraphs"),"Paragraph")
    run = node(node(par,"TextRuns"),"TextRun")
    node(run,"Value",value)
    style(run,FontFamily="Segoe UI",FontSize=font,Color=ink,FontWeight=weight)
    style(par,TextAlign=align)
    if pos:
        for tag, val in zip(("Left","Top","Width","Height"),pos):
            node(tb,tag,inch(val))
    st = style(tb,VerticalAlign="Middle",PaddingLeft="3pt",PaddingRight="2pt")
    if fill:
        node(st,"BackgroundColor",fill)
    return tb

ds = node(node(root,"DataSources"),"DataSource",Name="DeckData")
conn=node(ds,"ConnectionProperties")
node(conn,"DataProvider","ENTERDATA")
node(conn,"ConnectString","")
sets = node(root,"DataSets")
section = node(node(root,"ReportSections"),"ReportSection")
body = node(section,"Body")
items = node(body,"ReportItems")
total = 0
for idx, name in enumerate(names):
    rows = [r for r in data if r["view"] == name]
    cols = sorted({(int(r["position"]), r["category"]) for r in rows})
    decks = sorted({(int(r["deck_order"]), r["deck"], r["group"]) for r in rows})
    vals = {(r["deck"],r["category"]):(round(100*float(r["slot_share"]),1) if name=="Printed card colors" else int(r["count"])) for r in rows}
    assert len(decks)==20 and len(vals)==20*len(cols)
    total += len(vals)
    keys = ["Deck","Cohort"] + [f"V{i}" for i in range(len(cols))] + [f"Shade{i}" for i in range(len(cols))]
    dsname=f"Data{idx}"
    data_set=node(sets,"DataSet",Name=dsname)
    query=node(data_set,"Query")
    node(query,"DataSourceName","DeckData")
    xml=ET.Element("Query")
    node(xml,"ElementPath","Rows/Row")
    raw=node(node(xml,"XmlData"),"Rows")
    peak=max(vals.values()) or 1
    shades=["#F4F7F8","#D9ECE9","#B9DDD7","#8BC6C0","#67AEA9","#42958F"]
    for _,deck,group in decks:
        row=node(raw,"Row")
        node(row,"Deck",deck)
        node(row,"Cohort",group)
        for j,(_,category) in enumerate(cols):
            n = vals[deck,category]
            node(row,f"V{j}",f"{n}%" if name in ("Printed card colors","Casting-cost emphasis") else n)
            node(row,f"Shade{j}",shades[round(n*5/peak)])
    node(query,"CommandText",ET.tostring(xml,encoding="unicode"))
    fields=node(data_set,"Fields")
    for key in keys:
        node(node(fields,"Field",Name=key),"DataField",key)
    rect=node(items,"Rectangle",Name=f"Chart{idx}")
    inside=node(rect,"ReportItems")
    box(inside,name,"22pt",weight="Bold",pos=(0,0,15.8,.44))
    box(inside,units[name] + "   |   1 player deck and 19 public lists","10pt",pos=(0,.52,15.8,.33))
    grid=node(inside,"Tablix",Name=f"Matrix{idx}")
    tab=node(grid,"TablixBody")
    widths=[3.1,1.2]+[11.45/len(cols)]*len(cols)
    columns=node(tab,"TablixColumns")
    for w in widths:
        node(node(columns,"TablixColumn"),"Width",inch(w))
    table_rows=node(tab,"TablixRows")
    for header in (True,False):
        tr=node(table_rows,"TablixRow")
        node(tr,"Height",inch(.82 if header else .305))
        cells=node(tr,"TablixCells")
        for j, text in enumerate(["Deck","Group"]+[c for _,c in cols]):
            cc=node(node(cells,"TablixCell"),"CellContents")
            box(cc,text if header else f"=Fields!{keys[j]}.Value",
                font="8pt" if header or j<2 else "10pt",weight="Bold" if header or j>=2 else "Normal",
                align="Left" if j<2 else "Center",fill="#E6EDF1" if header else None if j<2 else f"=Fields!Shade{j-2}.Value")
    members=node(node(grid,"TablixColumnHierarchy"),"TablixMembers")
    for w in widths:
        node(members,"TablixMember")
    members=node(node(grid,"TablixRowHierarchy"),"TablixMembers")
    node(members,"TablixMember")
    node(node(members,"TablixMember"),"Group",Name=f"Details{idx}")
    node(grid,"DataSetName",dsname)
    node(grid,"Top",inch(1.03))
    node(grid,"Height",inch(1.125))
    node(grid,"Width",inch(sum(widths)))
    if idx<len(names)-1:
        node(node(rect,"PageBreak"),"BreakLocation","End")
    node(rect,"PageName",name)
    node(rect,"Top",inch(idx*2.20))
    node(rect,"Width",inch(15.8))
    node(rect,"Height",inch(2.20))
node(body,"Height",inch(len(names)*2.20))
node(section,"Width",inch(15.8))
page=node(section,"Page")
for tag,val in [("PageWidth",16.5),("PageHeight",9.2),("LeftMargin",.25),("RightMargin",.25),("TopMargin",.25),("BottomMargin",.25)]:
    node(page,tag,inch(val))
ET.ElementTree(root).write(here / "dashboards/power-bi/Commander Gameplay Comparison.rdl",encoding="utf-8",xml_declaration=True)
print(len(names),"report pages;",total,"chart values")
